# Laboratorio 1 - Ollama, chat y embeddings

**Duracion estimada:** 45 minutos

**Objetivo**

- Diferenciar entre un modelo de chat y un modelo de embeddings.
- Invocar ambos desde Python usando el stack del repositorio.
- Inspeccionar el tamano y el comportamiento basico de los vectores.

**Prerequisitos**

- `docker compose up -d`
- Modelos descargados con `scripts/pull-models.ps1` o `scripts/pull-models.sh`
- Entorno Python del repositorio instalado

**Criterios de exito**

- Obtienes una respuesta del modelo de chat.
- Generas embeddings para varias frases.
- Comparas al menos dos similitudes y justificas el resultado.


## Antes de empezar

Ejecuta la siguiente celda y verifica que:

1. El notebook encuentra `.env`.
2. Ves los puertos esperados.
3. Los nombres del modelo de chat y embeddings tienen sentido para tu entorno.


In [ ]:
import math
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"

print(
    {
        "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
        "CHAT_MODEL": CHAT_MODEL,
        "EMBEDDING_MODEL": EMBEDDING_MODEL,
    }
)


## Paso 1 - Instanciar los dos tipos de modelo

Completa la siguiente celda para crear:

- `chat_model` con `ChatOllama`
- `embedding_model` con `OllamaEmbeddings`

Usa los valores ya cargados desde `.env`.


In [ ]:
chat_model = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)
embedding_model = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print(type(chat_model).__name__, type(embedding_model).__name__)


### Checkpoint 1

Cuando completes el paso anterior, responde mentalmente:

- Que diferencia de responsabilidad tiene cada objeto?
- Cual de los dos genera texto y cual genera vectores?


## Paso 2 - Probar generacion con el modelo de chat

Pide una respuesta corta y controlada. Usa temperatura cero para tener un comportamiento mas estable.

Sugerencia de prompt:

`Resume en dos frases para que sirve un sistema RAG en local.`


In [ ]:
chat_response = chat_model.invoke(
    "Resume en dos frases para que sirve un sistema RAG en local."
).content

print(chat_response)


## Reflexion 1

**Respuesta orientativa**

- El modelo de chat se usa cuando queremos generar o transformar lenguaje natural.
- El modelo de embeddings se usa cuando queremos representar texto como vectores para compararlo o recuperarlo.
- `temperature=0` ayuda a que la respuesta sea mas estable y comparable entre ejecuciones.


## Paso 3 - Generar embeddings del dominio

Trabaja con frases de `FrasoHome` y genera un vector por frase.


In [ ]:
sentences = [
    "La politica de devoluciones permite comunicar danos de transporte en 72 horas.",
    "El manual de montaje recomienda no apretar totalmente los tornillos al principio.",
    "Qdrant almacena vectores y payload para recuperar contexto relevante.",
    "El cliente debe abrir una incidencia si faltan tornillos en la caja.",
]

vectors = embedding_model.embed_documents(sentences)

print(f"Numero de frases: {len(sentences)}")
print(f"Numero de vectores: {len(vectors)}")
print(f"Dimension del primer vector: {len(vectors[0])}")
print(vectors[0][:8])


## Paso 4 - Comparar similitud semantica

Implementa una similitud coseno sencilla y compara:

- frase 0 con frase 3
- frase 0 con frase 2

Explica cual par deberia parecerse mas y por que.


In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)


sim_0_3 = cosine_similarity(vectors[0], vectors[3])
sim_0_2 = cosine_similarity(vectors[0], vectors[2])

print({"sim_0_3": round(sim_0_3, 4), "sim_0_2": round(sim_0_2, 4)})


### Checkpoint 2

Verifica estos puntos:

- La dimension del embedding es consistente para todas las frases.
- La similitud entre frases del mismo dominio deberia ser mayor que entre frases menos relacionadas.
- Ya puedes explicar por que un embedding no responde preguntas por si solo.


## Reflexion 2

**Respuesta orientativa**

- La dimension del vector refleja el espacio numerico en el que el modelo representa el significado.
- Una mayor dimension no garantiza mejor retrieval, pero si condiciona la coleccion vectorial.
- Un embedding no responde preguntas por si solo: solo posiciona textos en un espacio semantico para compararlos.

**Mini extension opcional**

Anade una quinta frase del dominio y compara su similitud con las cuatro anteriores.
